In [ ]:
import pandas as pd 
import json
import pandas as pd
import bw2data as bd
import bw2calc as bc

# import own Python files, vars, mappings, and functions
from config import (CC_METHOD, NAME_REF_DB, NAME_FUTURE_DB,
                    COST_DATA, PROJECT_NAME, OUT_JSON_GHG, OUT_JSON_GHG_FUTURE)
import create_db_lca_functions as lcaf

14:41:54 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
GENERATE_NEW_LCA_DB=False
CALC_ALL_LCA_IMPACTS=True

In [3]:
PROJECT_NAME

'optimizer_nh3_bw25'

import brightway2 as bw
if 'biosphere3' in list(bw.databases):
    print("Deleting existing database...")
    del bw.databases['biosphere3']
bw.databases

In [4]:
bd.projects.set_current(PROJECT_NAME)

# If we want to use a full LCA approach, we have to set-up the LCA database:
if GENERATE_NEW_LCA_DB:
    bd.projects.set_current(PROJECT_NAME)
    # Import LCIA methods, import ecoinvent cut-off and consequential dbs
    #lcaf.import_additional_lcias()
    lcaf.import_ecoinvent_database()
    # Generate reference database with premise to import additional novel LCIs.
    lcaf.generate_reference_database()

    # Make future scenario, use 2C scenario from REMIND
    list_spec_scenarios, list_names = lcaf.generate_future_ei_dbs(scenarios = ["SSP2-PkBudg1000"], iam = 'remind',
                                       start_yr=2025, end_yr = 2050, step = 25, endstring="base")
    lcaf.generate_prospective_lca_dbs(list_spec_scenarios, list_names)

# Generate GHG emission data
if CALC_ALL_LCA_IMPACTS:
    cost_dict = COST_DATA[NAME_REF_DB].to_dict() # techno-economic data
    dict_ghg_impacts = lcaf.get_tech_environmental_burdens(cost_dict) 
    with open("input_data/dict_ghg_impacts.txt", 'w') as file:
        json.dump(dict_ghg_impacts, file)
        
    cost_dict_future = COST_DATA[NAME_FUTURE_DB].to_dict() # techno-economic data
    dict_ghg_impacts_future = lcaf.get_tech_environmental_burdens(cost_dict_future, sec_db=NAME_FUTURE_DB) 
    with open("input_data/dict_ghg_impacts_future.txt", 'w') as file:
        json.dump(dict_ghg_impacts_future, file)
else:
    #Otherwise, just used the stored data valid for ecoinvent cut-off
    with open("input_data/dict_ghg_impacts.txt", 'r') as file:
        dict_ghg_impacts = json.load(file)
        
    with open("input_data/dict_ghg_impacts_future.txt", 'r') as file:
        dict_ghg_impacts_future = json.load(file)
dict_ghg_impacts

{'ghg_imp_h2_ves': 17.04130906058737,
 'ghg_imp_pv': 1579.8958857231737,
 'ghg_imp_wind_on': 648.7297997691069,
 'ghg_imp_electr': 513.3507383589142,
 'ghg_imp_bat_cap': 136.03135919886256,
 'ghg_impact_grid_network': 102.28903146690008,
 'ghg_imp_asu': 0.00032177133637543477,
 'ghg_imp_hb': 0.057602371765460174}

In [5]:
dict_ghg_impacts_future

{'ghg_imp_h2_ves': 6.5074787351482,
 'ghg_imp_pv': 316.98510826548363,
 'ghg_imp_wind_on': 301.2385348433775,
 'ghg_imp_electr': 29.365497666746304,
 'ghg_imp_bat_cap': 30.003363612916782,
 'ghg_impact_grid_network': 26.17529236016685,
 'ghg_imp_asu': 6.962708526492677e-05,
 'ghg_imp_hb': 0.020867579095956316}

## Generate pre-calculated GHG emission factors for the grid

In [6]:
metadata_list = []
for db in [NAME_REF_DB, NAME_FUTURE_DB]:
    all_acts = [act for act in bd.Database(db) if ('market for electricity, low voltage' == act['name'] or 'market group for electricity, low voltage' == act['name']) and 'electricity, low voltage' == act['reference product'] ] 
    
    for act_sel in all_acts:
        """Store metadata"""
        metadata = {
            'location': act_sel.get('location', ''),
            "db": db,
            'name': act_sel.get('name', ''),
            'unit': act_sel.get('unit', ''),
            'reference_product': act_sel.get('reference product', ''),
            'key': act_sel.key,
            
        }
        metadata_list.append(metadata)

df_meta = pd.DataFrame(metadata_list)
df_meta

,location,db,name,unit,reference_product,key
0,AM,ecoinvent_312_reference,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_312_reference, 228e245d0e774700b256..."
1,SG,ecoinvent_312_reference,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_312_reference, 4323fd4563294be6bf63..."
2,HK,ecoinvent_312_reference,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_312_reference, cbbded75940b4cda9c20..."
3,CN-NWG,ecoinvent_312_reference,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_312_reference, 9d0d8c505b604840b5aa..."
4,US,ecoinvent_312_reference,"market group for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_312_reference, 88b4341f69534f90bfe8..."
...,...,...,...,...,...,...
392,BH,ecoinvent_remind_SSP2-PkBudg1000_2050_base,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_remind_SSP2-PkBudg1000_2050_base, 5..."
393,BR-Southern grid,ecoinvent_remind_SSP2-PkBudg1000_2050_base,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_remind_SSP2-PkBudg1000_2050_base, 1..."
394,NA,ecoinvent_remind_SSP2-PkBudg1000_2050_base,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_remind_SSP2-PkBudg1000_2050_base, c..."
395,NL,ecoinvent_remind_SSP2-PkBudg1000_2050_base,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_remind_SSP2-PkBudg1000_2050_base, 0..."


In [ ]:

def run_mlca(
    activity_keys: list,
    functional_units: list,
    result_index_labels: list,
    column_suffix: str,
    impact_methods: list,
) -> pd.DataFrame:
    """
    Run a BW2.5 MultiLCA calculation and return a DataFrame of results.

    Parameters
    ----------
    activity_keys : list
        Brightway activity references. Can be:
        - integer node ids
        - (database, code) tuples
        - Activity objects
    functional_units : list
        Functional unit amounts.
    result_index_labels : list
        Labels to keep in the output table, e.g. Brightway keys.
    column_suffix : str
        Suffix for result columns.
    impact_methods : list
        List of LCIA method tuples.

    Returns
    -------
    pd.DataFrame
        DataFrame with a normal 'key' column, not an index.
    """
    if not (len(activity_keys) == len(functional_units) == len(result_index_labels)):
        raise ValueError(
            "activity_keys, functional_units, and result_index_labels must have the same length."
        )

    def to_node_id(obj):
        if isinstance(obj, int):
            return obj
        elif isinstance(obj, tuple) and len(obj) == 2:
            return bd.get_node(database=obj[0], code=obj[1]).id
        elif hasattr(obj, "id"):
            return obj.id
        else:
            raise TypeError(
                f"Unsupported activity key type: {type(obj)}. "
                "Use int ids, (database, code) tuples, or Activity objects."
            )

    # safe internal labels for MultiLCA
    demand_labels = [f"fu_{i}" for i in range(len(activity_keys))]

    demands = {
        demand_label: {to_node_id(key): float(fu)}
        for demand_label, key, fu in zip(demand_labels, activity_keys, functional_units)
    }

    method_config = {"impact_categories": impact_methods}

    data_objs = bd.get_multilca_data_objs(
        functional_units=demands,
        method_config=method_config,
    )

    mlca = bc.MultiLCA(
        demands=demands,
        method_config=method_config,
        data_objs=data_objs,
    )
    mlca.lci()
    mlca.lcia()

    suffix = column_suffix if (column_suffix == "" or column_suffix.startswith("_")) else f"_{column_suffix}"
    colnames = [f"lca_impact{suffix}_{method[-1]}" for method in impact_methods]

    rows = []
    for i, demand_label in enumerate(demand_labels):
        row = {"key": result_index_labels[i]}
        for method in impact_methods:
            col = f"lca_impact{suffix}_{method[-1]}"
            row[col] = mlca.scores[(method, demand_label)]
        rows.append(row)

    return pd.DataFrame(rows)

In [28]:
CC_METHOD

('ecoinvent-3.12',
 'IPCC 2021 (incl. biogenic CO2)',
 'climate change: total (incl. biogenic CO2, incl. SLCFs)',
 'global warming potential (GWP100)')

In [30]:
def run_mlca(
    activity_keys: list,
    functional_units: list,
    result_index_labels: list,
    column_suffix: str,
    impact_methods: list,
) -> pd.DataFrame:
    """
    Run a BW2.5 MultiLCA calculation and return a DataFrame of results.

    Parameters
    ----------
    activity_keys : list
        Brightway activity references. Can be:
        - integer node ids
        - (database, code) tuples
        - Activity objects
    functional_units : list
        Functional unit amounts.
    result_index_labels : list
        Labels to keep in the output table, e.g. Brightway keys.
    column_suffix : str
        Suffix for result columns.
    impact_methods : list
        List of LCIA method tuples.

    Returns
    -------
    pd.DataFrame
        DataFrame with a normal 'key' column, not an index.
    """
    if not (len(activity_keys) == len(functional_units) == len(result_index_labels)):
        raise ValueError(
            "activity_keys, functional_units, and result_index_labels must have the same length."
        )

    def to_node_id(obj):
        if isinstance(obj, int):
            return obj
        elif isinstance(obj, tuple) and len(obj) == 2:
            return bd.get_node(database=obj[0], code=obj[1]).id
        elif hasattr(obj, "id"):
            return obj.id
        else:
            raise TypeError(
                f"Unsupported activity key type: {type(obj)}. "
                "Use int ids, (database, code) tuples, or Activity objects."
            )

    # safe internal labels for MultiLCA
    demand_labels = [f"fu_{i}" for i in range(len(activity_keys))]

    demands = {
        demand_label: {to_node_id(key): float(fu)}
        for demand_label, key, fu in zip(demand_labels, activity_keys, functional_units)
    }

    method_config = {"impact_categories": impact_methods}

    data_objs = bd.get_multilca_data_objs(
        functional_units=demands,
        method_config=method_config,
    )

    mlca = bc.MultiLCA(
        demands=demands,
        method_config=method_config,
        data_objs=data_objs,
    )
    mlca.lci()
    mlca.lcia()

    suffix = column_suffix if (column_suffix == "" or column_suffix.startswith("_")) else f"_{column_suffix}"
    colnames = [f"lca_impact{suffix}_{method[-1]}" for method in impact_methods]

    rows = []
    for i, demand_label in enumerate(demand_labels):
        row = {"key": result_index_labels[i]}
        for method in impact_methods:
            col = f"lca_impact{suffix}_{method[-1]}"
            row[col] = mlca.scores[(method, demand_label)]
        rows.append(row)

    return pd.DataFrame(rows)

df_lca = run_mlca(
    activity_keys=df_meta["key"].tolist(),
    functional_units=[1] * len(df_meta),
    result_index_labels=df_meta["key"].tolist(),
    column_suffix="",
    impact_methods=[CC_METHOD]
)

df_total_power = df_meta.merge(df_lca, on="key", how="left")
df_total_power

,location,db,name,unit,reference_product,key,lca_impact_global warming potential (GWP100)
0,AM,ecoinvent_312_reference,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_312_reference, 228e245d0e774700b256...",0.349001
1,SG,ecoinvent_312_reference,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_312_reference, 4323fd4563294be6bf63...",0.550334
2,HK,ecoinvent_312_reference,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_312_reference, cbbded75940b4cda9c20...",0.878719
3,CN-NWG,ecoinvent_312_reference,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_312_reference, 9d0d8c505b604840b5aa...",0.843093
4,US,ecoinvent_312_reference,"market group for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_312_reference, 88b4341f69534f90bfe8...",0.443643
...,...,...,...,...,...,...,...
392,BH,ecoinvent_remind_SSP2-PkBudg1000_2050_base,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_remind_SSP2-PkBudg1000_2050_base, 5...",0.026373
393,BR-Southern grid,ecoinvent_remind_SSP2-PkBudg1000_2050_base,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_remind_SSP2-PkBudg1000_2050_base, 1...",0.037876
394,NA,ecoinvent_remind_SSP2-PkBudg1000_2050_base,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_remind_SSP2-PkBudg1000_2050_base, c...",0.031526
395,NL,ecoinvent_remind_SSP2-PkBudg1000_2050_base,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage","(ecoinvent_remind_SSP2-PkBudg1000_2050_base, 0...",0.016445


In [31]:
df_total_power_2025=df_total_power[df_total_power['db'] == NAME_REF_DB]
df_total_power_2050=df_total_power[df_total_power['db'] == NAME_FUTURE_DB]

# Convert to dictionary indexed by 'location'
total_power_dict = df_total_power_2025.set_index('location').to_dict(orient='index')
total_power_dict_future = df_total_power_2050.set_index('location').to_dict(orient='index')

# Save to JSON file
with open(OUT_JSON_GHG, "w") as f:
    json.dump(total_power_dict, f, indent=2)

with open(OUT_JSON_GHG_FUTURE, "w") as f:
    json.dump(total_power_dict_future, f, indent=2)